In [1]:
#imports 

import torch
import torch.nn as nn
import math

In [2]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, len_seq, d_model, dropout):
        super().__init__()
        
        self.len_seq = len_seq
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(len_seq, d_model)
        pos = torch.arange(0, len_seq, dtype=torch.float).unsqueeze(1)
        
        exp_term = torch.arange(0, d_model, step=2) / d_model
        
        pe[:, 0::2] = torch.sin(pos / 1e4 ** exp_term)
        
        pe[:, 1::2] = torch.cos(pos / 1e4 ** exp_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pos_enc', pe)
        
    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad(False)
        return self.dropout(x)

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h, dropout):
        super().__init__()
        
        self.d_model = d_model
        self.h = h
        self.dropout = nn.Dropout(dropout)
        
        if d_model % h  != 0:
            raise ValueError('d_model is not divisible by h')
        
        self.d_k = d_model // h
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        
    def SelfAttention(self, X, mask, dropout):
        
        query = X
        key = X
        val = X
        
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)
        
        attention_scores = attention_scores.softmax(dim=-1)
        
        if dropout is not None:
            attention_scores  = dropout(attention_scores)
        
        
        return (attention_scores @ val), attention_scores
        
        
    def forward(self, x, mask):
        
        query = x
        key = x
        val = x
        
        
        query = self.w_q(x)
        key = self.w_k(x)
        val = self.w_v(x)
        
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)     
        val = val.view(val.shape[0], val.shape[1], self.h, self.d_k).transpose(1, 2)  
        
    
        x, self.attention_scores = self.SelfAttention(x, self.dropout)
                
        x = x.transpose(1, 2).contiguos().view(x.shape[0], x.shape[1], self.h * self.d_k)
        
        return self.w_o(x)

In [5]:
class FeedForwardNet(nn.Module):
    def __init__(self, d_model, dff, droput):
        super().__init__()
        
        self.l1 = nn.Linear(d_model, dff)
        self.l2 = nn.Linear(dff, d_model)
        self.dropout = nn.Dropout(droput)
        
    def forward(self, x):
            
        x = nn.ReLU(self.l1(x))
        x = self.dropout(x)
        x = self.l2(x)
        
        return x

In [6]:
class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.epsilon = 1e-6
        
    def forward(self, x):
        
        mean = x.mean(dim= -1, keepdim=True)
        
        std = x.std(dim= -1, keepdim=True)
        
        
        return self.alpha * (x - mean) / (std + self.epsilon) + self.beta

In [7]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, multi_head_attention_block, feed_forward_block, dropout):
        super().__init__()
        self.multi_head_attention_block = multi_head_attention_block
        self.feed_forward_block = feed_forward_block
        
 
        self.norm1 = LayerNorm(d_model) 
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        
        attention_out = self.multi_head_attention_block(x, src_mask)
        
        x = self.norm1(x + self.dropout(attention_out))
      
        ff_out = self.feed_forward_block(x)
        
        x = self.norm2(x + self.dropout(ff_out))

        return x

In [8]:
import copy

class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
    
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
 
        self.norm = LayerNorm(layer.norm1.alpha.shape[0])

    def forward(self, x, mask):
 
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)